In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DATA_PATH = Path(
    r"C:\Users\ak789\Desktop\proj\SentinelPay-AI\data\raw\PS_20174392719_1491204439457_log.csv"
)

OUTPUT_DIR = Path(
    r"C:\Users\ak789\Desktop\proj\SentinelPay-AI\data\processed"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Starting SentinelPay AI - Exploratory Data Analysis")

# Aggregated statistics
type_stats = {}
fraud_amount = {0: {"sum": 0, "count": 0, "max": 0},
                1: {"sum": 0, "count": 0, "max": 0}}

balance_stats = {
    0: {"origin_error": 0, "dest_error": 0, "count": 0},
    1: {"origin_error": 0, "dest_error": 0, "count": 0}
}

for chunk in pd.read_csv(DATA_PATH, chunksize=500_000):

    # Balance inconsistencies
    chunk["origin_error"] = (
        chunk["oldbalanceOrg"] - chunk["amount"]
        - chunk["newbalanceOrig"]
    ).abs()

    chunk["dest_error"] = (
        chunk["oldbalanceDest"] + chunk["amount"]
        - chunk["newbalanceDest"]
    ).abs()

    # Transaction type and fraud statistics
    grouped = chunk.groupby(["type", "isFraud"]).agg(
        count=("amount", "count"),
        amount_sum=("amount", "sum"),
        amount_max=("amount", "max")
    )

    for (transaction_type, fraud), row in grouped.iterrows():
        key = (transaction_type, fraud)

        if key not in type_stats:
            type_stats[key] = {
                "count": 0,
                "amount_sum": 0,
                "amount_max": 0
            }

        type_stats[key]["count"] += row["count"]
        type_stats[key]["amount_sum"] += row["amount_sum"]
        type_stats[key]["amount_max"] = max(
            type_stats[key]["amount_max"], row["amount_max"]
        )

    # Amount statistics by fraud status
    for fraud in [0, 1]:
        subset = chunk[chunk["isFraud"] == fraud]

        if not subset.empty:
            fraud_amount[fraud]["sum"] += subset["amount"].sum()
            fraud_amount[fraud]["count"] += len(subset)
            fraud_amount[fraud]["max"] = max(
                fraud_amount[fraud]["max"],
                subset["amount"].max()
            )

            balance_stats[fraud]["origin_error"] += (
                subset["origin_error"].sum()
            )
            balance_stats[fraud]["dest_error"] += (
                subset["dest_error"].sum()
            )
            balance_stats[fraud]["count"] += len(subset)

# Build summary report
records = []

for (transaction_type, fraud), stats in type_stats.items():
    records.append({
        "transaction_type": transaction_type,
        "isFraud": fraud,
        "transaction_count": stats["count"],
        "total_amount": stats["amount_sum"],
        "average_amount": stats["amount_sum"] / stats["count"],
        "max_amount": stats["amount_max"]
    })

summary = pd.DataFrame(records)

# Calculate fraud rate for each transaction type
type_summary = summary.pivot(
    index="transaction_type",
    columns="isFraud",
    values="transaction_count"
).fillna(0)

type_summary.columns = ["Legitimate", "Fraudulent"]
type_summary["Total"] = (
    type_summary["Legitimate"] + type_summary["Fraudulent"]
)
type_summary["Fraud_Rate_Percent"] = (
    type_summary["Fraudulent"] / type_summary["Total"] * 100
)

print("\n========== FRAUD BY TRANSACTION TYPE ==========")
print(type_summary.round(4))

# Average and maximum transaction amounts
print("\n========== AMOUNT ANALYSIS ==========")

for fraud, label in [(0, "Legitimate"), (1, "Fraudulent")]:
    stats = fraud_amount[fraud]

    average = stats["sum"] / stats["count"]

    print(f"\n{label}:")
    print("Average amount:", round(average, 2))
    print("Maximum amount:", round(stats["max"], 2))

# Balance error analysis
print("\n========== BALANCE ERROR ANALYSIS ==========")

for fraud, label in [(0, "Legitimate"), (1, "Fraudulent")]:
    stats = balance_stats[fraud]
    count = stats["count"]

    print(f"\n{label}:")
    print("Average origin balance error:",
          round(stats["origin_error"] / count, 2))
    print("Average destination balance error:",
          round(stats["dest_error"] / count, 2))

# Save reports
summary.to_csv(
    OUTPUT_DIR / "fraud_type_summary.csv",
    index=False
)

type_summary.to_csv(
    OUTPUT_DIR / "fraud_rate_by_type.csv"
)

# Visualization 1: Fraud count by transaction type
plt.figure(figsize=(10, 5))
type_summary[["Legitimate", "Fraudulent"]].plot(
    kind="bar",
    figsize=(10, 5)
)
plt.title("Transaction Distribution by Type and Fraud Status")
plt.xlabel("Transaction Type")
plt.ylabel("Number of Transactions")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "transaction_distribution.png")
plt.close()

# Visualization 2: Fraud rate by transaction type
plt.figure(figsize=(9, 5))
type_summary["Fraud_Rate_Percent"].plot(
    kind="bar"
)
plt.title("Fraud Rate by Transaction Type")
plt.xlabel("Transaction Type")
plt.ylabel("Fraud Rate (%)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fraud_rate_by_type.png")
plt.close()

print("\nEDA completed successfully!")
print("Reports saved to:", OUTPUT_DIR)
